In [3]:
import numpy as np
import pandas as pd
import copy
import os
from collections import namedtuple
from auto_cloze import REINDEX

## Compiling and sorting each round of cloze resulsts

In [ ]:
os.chdir('<path_to_response_sheets>')
rep_jan = pd.read_excel('Unexp_Rep.xlsx', sheet_name='unexp_rep_JAN', header=0)
rep_feb = pd.read_excel('Unexp_Rep.xlsx', sheet_name='unexp_rep_FEB', header=0)
rep_march = pd.read_excel('Unexp_Rep.xlsx', sheet_name='unexp_rep_MARCH', header=0)

jan = pd.read_excel('Ratings.xlsx', sheet_name='Jan_Cloze', header=0)
feb = pd.read_excel('Ratings.xlsx', sheet_name='Feb_Cloze', header=0)
march = pd.read_excel('Ratings.xlsx', sheet_name='March_Cloze', header=0)
df = pd.DataFrame(None, None, pd.concat((jan, feb, march), axis=0).columns)
for i, x in enumerate(pd.concat((jan, feb, march), axis=0).index):
    df.loc[len(df)] = pd.concat((jan, feb, march), axis=0).iloc[i]

reindex = REINDEX(df['Constraint'].to_numpy())
df = df.reindex(reindex)

mapper = {}
for i, x in enumerate(df.index):
    mapper[x] = i
df = df.rename(mapper=mapper, axis=0)


rep_check = np.empty(len(df), dtype=object)
for i, x in enumerate(df.index):
    if df['Sframes'][i] in rep_jan['Sframes'].to_numpy(): rep_check[i] = 'Jan'
    elif df['Sframes'][i] in rep_feb['Sframes'].to_numpy(): rep_check[i] = 'Feb'
    elif df['Sframes'][i] in rep_march['Sframes'].to_numpy(): rep_check[i] = 'March'
df.insert(6, 'Unexp_Rep', rep_check)
df.drop(['unexp_as_best', 'unexp_as_alt'], axis=1, inplace=True)

In [ ]:
condition = np.empty(len(df), dtype=object)
for i, x in enumerate(df.index):
    if df['Constraint'][i] >= 0.7 and 'Neu' in df['Condition'][i]: condition[i] = 'Neu_SC'
    elif df['Constraint'][i] <= 0.4 and 'Neu' in df['Condition'][i]: condition[i] = 'Neu_WC'
    elif df['Constraint'][i] >= 0.7 and 'Emo' in df['Condition'][i]: condition[i] = 'Emo_SC'
    elif df['Constraint'][i] <= 0.4 and 'Emo' in df['Condition'][i]: condition[i] = 'Emo_WC'
    elif 0.4 < df['Constraint'][i] < 0.7 and 'Neu' in df['Condition'][i]: condition[i] = 'Neu_NA'
    elif 0.4 < df['Constraint'][i] < 0.7 and 'Emo' in df['Condition'][i]: condition[i] = 'Emo_NA'
    
df.insert(0, 'Rated_Condition', condition)     
df.drop('Condition', axis=1, inplace=True)

In [157]:
wc = pd.DataFrame(None, None, df.columns)
sc = pd.DataFrame(None, None, df.columns)
na = pd.DataFrame(None, None, df.columns)

for i, x in enumerate(df['Rated_Condition']):
    if 'WC' in x: wc.loc[len(wc.index)] = df.iloc[i]
    elif 'SC' in x: sc.loc[len(sc.index)] = df.iloc[i]
    elif 'NA' in x: na.loc[len(na.index)] = df.iloc[i]

## Main paring algorithm

In [161]:
df1 = pd.DataFrame(None, None, wc.columns)
df2 = pd.DataFrame(None, None, wc.columns)
for i, x in enumerate(wc['Rated_Condition']):
    if 'Neu' in x: df1.loc[len(df1.index)] = wc.iloc[i]
    elif 'Emo' in x: df2.loc[len(df2.index)] = wc.iloc[i]

unexp = []
for i in wc['Unexpected']:
    if i not in unexp: unexp.append(i)
        
neu, emo = pd.DataFrame(None, None, wc.columns), pd.DataFrame(None, None, wc.columns)   
L = 1
while L <= 10:
    mapper1, mapper2 = {}, {}
    for i, x in enumerate(df1.index):
        mapper1[x] = i
    df1 = df1.rename(mapper=mapper1, axis=0)
    for i, x in enumerate(df2.index):
        mapper2[x] = i
    df2 = df2.rename(mapper=mapper2, axis=0)
    
    pair1, pair2 = [], []
    for i in range(len(unexp)):
        pair1.append([])
        pair2.append([])

    for i, x in enumerate(unexp):
        for j, y in enumerate(df1['Unexpected']):
            if y == x:
                pair1[i].append(j)
        for j, y in enumerate(df2['Unexpected']):
            if y == x:
                pair2[i].append(j)
                
    df1_pair, df2_pair = pd.DataFrame(None, None, wc.columns), pd.DataFrame(None, None, wc.columns)
    print(df1_pair)
    for i, x in enumerate(unexp):
        if len(pair1[i]) == L and len(pair2[i]) == L:
            df1_pair = pd.concat((df1_pair, df1.loc[pair1[i]]), axis=0)
            df1.drop(pair1[i], inplace=True)
            df2_pair = pd.concat((df2_pair, df2.loc[pair2[i]]), axis=0)
            df2.drop(pair2[i], inplace=True)
    neu = pd.concat((neu, df1_pair), axis=0)
    emo = pd.concat((emo, df2_pair), axis=0)
    print(len(df1.index))
    L += 1
    

Empty DataFrame
Columns: [Rated_Condition, Sframes, Best1, Best2, Best3, Best4, Unexp_Rep, Best5, Best6, Best7, Best8, Best9, Best10, Best11, Best12, Best13, Best14, Best15, Best16, Best17, Best18, Best19, Best20, Constraint, Optimal, Unexpected]
Index: []

[0 rows x 26 columns]
60
Empty DataFrame
Columns: [Rated_Condition, Sframes, Best1, Best2, Best3, Best4, Unexp_Rep, Best5, Best6, Best7, Best8, Best9, Best10, Best11, Best12, Best13, Best14, Best15, Best16, Best17, Best18, Best19, Best20, Constraint, Optimal, Unexpected]
Index: []

[0 rows x 26 columns]
60
Empty DataFrame
Columns: [Rated_Condition, Sframes, Best1, Best2, Best3, Best4, Unexp_Rep, Best5, Best6, Best7, Best8, Best9, Best10, Best11, Best12, Best13, Best14, Best15, Best16, Best17, Best18, Best19, Best20, Constraint, Optimal, Unexpected]
Index: []

[0 rows x 26 columns]
60
Empty DataFrame
Columns: [Rated_Condition, Sframes, Best1, Best2, Best3, Best4, Unexp_Rep, Best5, Best6, Best7, Best8, Best9, Best10, Best11, Best12, B

In [164]:
unexp1 = []
for i in df1['Unexpected']:
    if i not in unexp1: unexp1.append(i)
        
df1_sort = pd.DataFrame(None, None, wc.columns)
for i, x in enumerate(unexp1):
    for j, y in enumerate(df1['Unexpected']):
        if y == x: df1_sort.loc[len(df1_sort)] = df1.loc[j]

df2_sort = pd.DataFrame(None, None, wc.columns)
for i, x in enumerate(unexp1):
    mapper = {}
    for j, y in enumerate(df2.index):
        mapper[y] = j
    df2.rename(mapper=mapper, axis=0, inplace=True)
    for k, z in enumerate(df2['Unexpected']):
        if z == x:
            df2_sort.loc[len(df2_sort)] = df2.loc[k]
            df2.drop([k], inplace=True) 
            
unexp2 = []
for i in df2_sort['Unexpected']:
    if i not in unexp2: unexp2.append(i)
print(len(unexp2))

df1 = pd.DataFrame(None, None, df.columns)
drop = []
for i, x in enumerate(df1_sort['Unexpected']):
    if x not in unexp2:
        df1.loc[len(df1)] = df1_sort.loc[i]
        drop.append(i)
df1_sort.drop(drop, inplace=True)
mapper = {}
for i, x in enumerate(df1_sort.index):
    mapper[x] = i
df1_sort.rename(mapper=mapper, inplace=True)

0


In [165]:
# neu and emo are fully paired
# df1_sort and df2_sort have the same set of unexpected endings
# but the number of sentences completed by each unexpected ening differ
# df1 and df2 have different unexpected endings
# unexp2 is the list of unexpected shared by df1_sort and df2_sort

count1, count2 = [], []
for i, x in enumerate(unexp2):
    c1 = 0
    for j, y in enumerate(df1_sort['Unexpected']):
        if y == x: c1 += 1
    count1.append(c1)
    c2 = 0
    for j, y in enumerate(df2_sort['Unexpected']):
        if y == x: c2 += 1
    count2.append(c2)
    
count = []
for i in range(len(count1)):
    count.append(max(count1[i], count2[i]))

sub_df1 = pd.DataFrame(None, index=np.arange(0, sum(count)), columns=wc.columns)
sub_df2 = pd.DataFrame(None, index=np.arange(0, sum(count)), columns=wc.columns)
for i in range(len(count)):
    loc1 = np.arange(sum(count[0:i]), sum(count[0:i+1])-(count[i]-count1[i]))
    loc2 = np.arange(sum(count[0:i]), sum(count[0:i+1])-(count[i]-count2[i]))
    sub_df1.iloc[loc1] = df1_sort.iloc[np.arange(sum(count1[0:i]), sum(count1[0:i+1]))]
    sub_df2.iloc[loc2] = df2_sort.iloc[np.arange(sum(count2[0:i]), sum(count2[0:i+1]))]

In [166]:
sub_df1

,Rated_Condition,Sframes,Best1,Best2,Best3,Best4,Unexp_Rep,Best5,Best6,Best7,...,Best14,Best15,Best16,Best17,Best18,Best19,Best20,Constraint,Optimal,Unexpected


In [167]:
sub_df2

,Rated_Condition,Sframes,Best1,Best2,Best3,Best4,Unexp_Rep,Best5,Best6,Best7,...,Best14,Best15,Best16,Best17,Best18,Best19,Best20,Constraint,Optimal,Unexpected


In [168]:
paired_neu = pd.concat((neu, sub_df1), axis=0)
paired_emo = pd.concat((emo, sub_df2), axis=0)

mapper1, mapper2 = {}, {}
for i, x in enumerate(paired_neu.columns):
    mapper1[x] = x + '.1'
    mapper2[x] = x + '.2'

index = np.arange(0, len(paired_neu))
paired_neu.rename(mapper=mapper1, axis=1, inplace=True)
paired_emo.rename(mapper=mapper2, axis=1, inplace=True)
paired_neu.index = index
paired_emo.index = index

In [169]:
paired_neu

,Rated_Condition.1,Sframes.1,Best1.1,Best2.1,Best3.1,Best4.1,Unexp_Rep.1,Best5.1,Best6.1,Best7.1,...,Best14.1,Best15.1,Best16.1,Best17.1,Best18.1,Best19.1,Best20.1,Constraint.1,Optimal.1,Unexpected.1


In [170]:
paired_emo

,Rated_Condition.2,Sframes.2,Best1.2,Best2.2,Best3.2,Best4.2,Unexp_Rep.2,Best5.2,Best6.2,Best7.2,...,Best14.2,Best15.2,Best16.2,Best17.2,Best18.2,Best19.2,Best20.2,Constraint.2,Optimal.2,Unexpected.2


In [171]:
paired_all = pd.concat((paired_neu, paired_emo), axis=1)
paired_all

,Rated_Condition.1,Sframes.1,Best1.1,Best2.1,Best3.1,Best4.1,Unexp_Rep.1,Best5.1,Best6.1,Best7.1,...,Best14.2,Best15.2,Best16.2,Best17.2,Best18.2,Best19.2,Best20.2,Constraint.2,Optimal.2,Unexpected.2


In [172]:
paired_all.to_excel('SEP_WC_Pairs.xlsx')
df1.to_excel('SEP_Unpaired_Neu_WC.xlsx')
df2.to_excel('SEP_Unpaired_Emo_WC.xlsx')